In [ ]:
cd ..

In [ ]:

from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
from pyspark.sql.functions import pandas_udf
import src.utils.config as config
from src.ingestion.last_activity_generator import generate_last_activity
from prepare_data_inference import prepare_data_inference
from pyspark.ml.functions import vector_to_array


In [ ]:
spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()

player_behavior = spark.read.parquet("./data/gold/player_behavior")
player_snapshot = spark.read.parquet("./data/gold/player_snapshot")
labels = spark.read.parquet("./data/gold/labels")

In [ ]:
players_silver = spark.read.parquet("./data/silver/players")
sessions_silver = spark.read.parquet("./data/silver/sessions")
transactions_silver = spark.read.parquet("./data/silver/transactions")
silver_money_events = spark.read.parquet("./data/silver/money_events")

players_silver = players_silver.drop('player_id')
transactions_silver = transactions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
sessions_silver = sessions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
silver_money_events = silver_money_events.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')

In [ ]:
test_date = '2024-05-20'
data_inference = prepare_data_inference(test_date)
data_inference.show(3)

In [ ]:
def compare(df1,df2): 
   return (( df1.exceptAll(df2).count() == 0) & (df2.exceptAll(df1).count() == 0))

m1 = player_behavior.filter(F.col('reference_date')==test_date).select('player_idx').join(data_inference, how='inner', on='player_idx', )
compare(m1, data_inference)

In [36]:
import mlflow 



spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()

sample_fraction = 1.0
mlflow.set_experiment("batch-mode experiment")

model_uri = 'runs:/1651abbb4a0e42c3ab0a48526599ba03/spark_model'
loaded_model = mlflow.spark.load_model(model_uri)

data_inference_ml = (player_behavior.select('player_idx').filter(F.col('reference_date')==test_date)
.join(data_inference, how ='inner', on='player_idx')
.join(player_snapshot, on="player_idx", how="inner")
.drop('reference_date')
)


with mlflow.start_run(run_name=test_date):

    #mlflow.set_tag("train_run_id", train_run_id)
    test_preds = loaded_model.transform(data_inference_ml).withColumn("p_churn", vector_to_array("probability")[1])

    



2026/03/04 20:46:48 INFO mlflow.spark: URI 'runs:/1651abbb4a0e42c3ab0a48526599ba03/spark_model/sparkml' does not point to the current DFS.
2026/03/04 20:46:48 INFO mlflow.spark: File 'runs:/1651abbb4a0e42c3ab0a48526599ba03/spark_model/sparkml' not found on DFS. Will attempt to upload the file.


In [39]:
test_preds.filter(F.col('prediction')==1.0).show()

+----------+--------------------+---------------------+------------------+------------------+----------------------+-----------------+--------------------+----------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+-------+----------+-----------+--------------+-------------+--------------+--------------------+-----------------------+--------------------+--------------------+--------------------+----------+------------------+
|player_idx|net_amount_result_7d|net_amount_result_30d|   balance_30d_ago|    balance_7d_ago|failed_withdrawals_30d|deposit_count_30d|withdrawal_count_30d|withdrawal_ratio|num_sessions_7d|net_game_result_7d|num_sessions_30d|avg_sessions_duration_30d|avg_bet_amount_30d|net_game_result_30d|country|age_bucket|country_idx|age_bucket_idx|  country_ohe|age_bucket_ohe|    numeric_features|numeric_features_scaled|            features|       rawPrediction|         probability|prediction|           p_c

In [ ]:
player_snapshot = player_snapshot.select(
'player_idx', 'country', 'age_bucket', 
 )


(player_behavior.filter(F.col('reference_date')==test_date)
.join(data_inference, how ='inner', on='player_idx')
.join(player_snapshot, on="player_idx", how="inner")
).



In [ ]:
data_inference.columns


In [ ]:
data_inference.count()